# Predicting hyperspectral images with conditional GANs (modular version)

All classes and functions of the original notebook now live in the `hsi_cgan/` package next to this notebook
(see `README.md` for a file-by-file overview). This notebook only keeps the orchestration code, unchanged.

In [ ]:
import sys
# Make the `hsi_cgan` package importable. If the folder containing `hsi_cgan/` is not the working directory
# (e.g. on Colab/Drive), append its path here:
# sys.path.append('/content/drive/My Drive/<folder that contains hsi_cgan>')

## 0. Setup (installs, imports, Drive mount)

In [ ]:
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from datetime import datetime
from torch.utils.data import DataLoader, Dataset, TensorDataset, IterableDataset, get_worker_info
from torch.optim.lr_scheduler import StepLR
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter


import numpy as np
import os
import sys
import json

!pip3 install sympy --upgrade

if 'google.colab' in sys.modules and 'cv2' not in sys.modules:
  !pip3 install opencv-python

if 'google.colab' in sys.modules and 'torchmetrics' not in sys.modules:
  !pip3 install torchmetrics


import cv2

if 'google.colab' in sys.modules and 'spectral' not in sys.modules:
    !pip3 install spectral

from spectral import imshow, view_cube
import spectral.io.envi as envi

if 'google.colab' in sys.modules and 'tqdm' not in sys.modules:
    !pip3 install tqdm

%matplotlib inline
from tqdm import tqdm

if 'google.colab' in sys.modules and 'optuna' not in sys.modules:
    !pip3 install optuna

import optuna

if 'google.colab' in sys.modules and 'rasterio' not in sys.modules:
    !pip3 install rasterio

import rasterio

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# To avoid disconnection in Google Colab
# drive.mount('/content/drive', force_remount=True) # ctrl + shift + i
# function preventColabDisconnect() {
#     setInterval(() => {
#         console.log("Preventing disconnect...");
#         document.querySelector("colab-toolbar-button").click();
#     }, 300000); // 300,000 ms = 10 minutes
# }
# preventColabDisconnect();

## 1. Imports from the package
Replaces the former definitions cell (generators, discriminators, dataset, memmap builder, training loop, ...).
See `hsi_cgan/models`, `hsi_cgan/data`, `hsi_cgan/metrics`, `hsi_cgan/training`.

In [ ]:
# Definitions that used to live in this notebook now come from the `hsi_cgan` package.
from hsi_cgan.models.generators import (CNNGenerator, UNetGenerator, ResidualBlock, ResNetGenerator, GeneratorLarge,
                                     GeneratorLarge_MI_PCA, GeneratorLarge_bandwidth, GeneratorMLP)
from hsi_cgan.models.discriminators import DiscriminatorMLP, PatchGANDiscriminator, DiscriminatorLarge
from hsi_cgan.models.factory import creating_generator_and_discriminator, creating_loss_function, creating_optimizer
from hsi_cgan.data.memmap_builder import create_concatenated_memmaps
from hsi_cgan.data.dataset import HyperspectralDataset
from hsi_cgan.metrics.losses import rmse_loss, mrae_loss
from hsi_cgan.training.checkpoints import save_checkpoint, load_checkpoint, save_hyperparameters
from hsi_cgan.training.validation import validate
from hsi_cgan.training.trainer import train

## 2. Training block

In [ ]:
############################# ------------------ STARTING TRAINING BLOCK  ------------------#############################
# Load your data here
# Some settings for the procedure True or False
load_orig_images = False # If you want to loading in from the original images and sample from these images
load_all_images = False # There will be sampling from each images if yes otherwise only from 8 images
concatenate = False # concatenating all the samples in smaller arrays
concatenate_larger = False # concatenating all the samples in one big sample array for training and one for validation
borders = 10 # The amount of cells around the white tile reference


# hyperparameters
patch_size = 60
random_patches = 53 #53 #this determines the amount of samples taken from 1 images (the amount will be taken both for training and validation)

hyperparameters_save = False
shuffle_data_loader = True
bands_list = ['B-NIR'] #'MI', 'PCA', 'bandwidth', 'micasense', '4bands', '3bands'
# 'B-RE717', 'R-RE717', 'G-RE717', 'G-NIR', 'B-NIR', 'R-NIR', 'G531-NIR', 'R650-NIR', 'R-RE705', 'R-RE740', 'B-RE705', 'B-RE740', 'G-RE705', 'G-RE740', 'G531-RE', 'R650-RE'


generators_list = ['ResNetGenerator'] #'CNNGenerator', 'ResNetGenerator','UNetGenerator', 'GeneratorLarge', 'MLP'


dictionary_list = {'CNNGenerator':[0.001, 'rgb', 'PatchGANDiscriminator'],
                   'UNetGenerator':[0.001, 'rgb', 'PatchGANDiscriminator'],
                   'ResNetGenerator':[0.0001, 'rgb', 'PatchGANDiscriminator'],
                   'GeneratorLarge':[0.0001, 'MLP', 'DiscriminatorLarge'],
                   'MLP':[0.001, 'MLP', 'MLP']
                   }

for band in bands_list:
  print('The current bands:', band)
  for generator in generators_list:
    print('The current generator:', generator)
    hyperparameters = {
        'load_model': False, #TODO
        'hard_code_known': False,
        'data_percentage': 1, # Allows to set the percentage of data being used in training
        'num_workers': 2,
        'Bands': band, # MI, PCA, bandwidth, micasense
        'learning_rate': dictionary_list[generator][0],
        'betas': (0.5, 0.99), #standard (0.5, 0.999)
        'batch_size': 16,
        'epochs': 100,
        'architecture': dictionary_list[generator][1], #options: rgb, cramer, MLP
        'name_loss': 'L1', #options: MSE, L1, BCE, CrossEntropy, SmoothL1Loss, KLDivLoss, HingeEmbeddingLoss, MarginRankingLoss, MultiLabelSoftMarginLoss, BCEWithLogitsLoss
        'name_adversarial_loss': 'BCEWithLogitsLoss',
        'name_g': generator, #options: CNNGenerator, UNetGenerator, ResNetGenerator, GeneratorLarge, MLP
        'name_d': dictionary_list[generator][2], #options: PatchGANDiscriminator, DiscriminatorLarge, MLP
        'name_optim': 'Adam', #options: Adam, SGD, RMSprop, Adagrad, Adadelta,    AdamDecay, RMSpropDecay
        'checkpoint_epoch': 88,
        'saving_epoch': 3,
        'basedir': '/content/drive/My Drive/AAA_thesis_twee',
        'decay_epochs': 25, #stepsize
        'decay_rate': 0.75, #gamma
        'exclude_begin': 35, #35
        'exclude_end': 40, #
    }
    temp = hyperparameters['checkpoint_epoch']
    hyperparameters['checkpoint_path'] = f'checkpoint_epoch_{temp}_'

    hyperparameters['name'] = hyperparameters['name_g'] + '_' + hyperparameters['name_d'] + \
    '_' + hyperparameters['name_optim'] + '_' + hyperparameters['name_loss'] + '_' + hyperparameters['name_adversarial_loss']

    hyperparameters['checkpoint_path'] += hyperparameters['name'] + '_{0}.pth'.format(hyperparameters['Bands'])

    # if not os.path.exists(hyperparameters['basedir']+'/checkpoints'):
    #     os.makedirs(hyperparameters['basedir']+'/checkpoints')

    # if not os.path.exists(hyperparameters['basedir']+'/memmaps'):
    #     os.makedirs(hyperparameters['basedir']+'/memmaps')

    # if not os.path.exists(hyperparameters['basedir']+'/models'):
    #     os.makedirs(hyperparameters['basedir']+'/models')

    # if not os.path.exists(hyperparameters['basedir']+'/hyperparameters'):
    #     os.makedirs(hyperparameters['basedir']+'/hyperparameters')

    # if not os.path.exists(hyperparameters['basedir']+'/data'):
    #     os.makedirs(hyperparameters['basedir']+'/data')

    wavelengths= np.load(r"{0}/wavelengthes.npy".format(hyperparameters['basedir']))
    hyperparameters['total_length'] = len(wavelengths)
    wavelengths =wavelengths[hyperparameters['exclude_begin']:hyperparameters['total_length']-hyperparameters['exclude_end']]

    if hyperparameters['Bands'] == "micasense":
      centers = {
        "R": 668,
        "G": 560,
        "B": 475,
        "R_edge": 717,
        "NIR": 842
        }
      closest_wavelengths = []


      for band, center in centers.items():

          idx = np.argmin(np.abs(wavelengths - center))

          closest_wavelengths.append(idx)

      indices = np.array(closest_wavelengths)


    elif hyperparameters['Bands'] == "R650-RE":
      centers = {
        "R": 650,
        # "G": 531,
        # "B": 475,
        "R_edge": 717,
        # "NIR": 842
        }
      closest_wavelengths = []


      for band, center in centers.items():

          idx = np.argmin(np.abs(wavelengths - center))

          closest_wavelengths.append(idx)

      indices = np.array(closest_wavelengths)




    elif hyperparameters['Bands'] == "G531-RE":
      centers = {
        # "R": 668,
        "G": 531,
        # "B": 475,
        "R_edge": 717,
        # "NIR": 842
        }
      closest_wavelengths = []


      for band, center in centers.items():

          idx = np.argmin(np.abs(wavelengths - center))

          closest_wavelengths.append(idx)

      indices = np.array(closest_wavelengths)



    elif hyperparameters['Bands'] == "B-RE740":
      centers = {
        # "R": 668,
        # "G": 560,
        "B": 475,
        "R_edge": 740,
        # "NIR": 842
        }
      closest_wavelengths = []


      for band, center in centers.items():

          idx = np.argmin(np.abs(wavelengths - center))

          closest_wavelengths.append(idx)

      indices = np.array(closest_wavelengths)


    elif hyperparameters['Bands'] == "B-RE705":
      centers = {
        # "R": 668,
        # "G": 560,
        "B": 475,
        "R_edge": 705,
        # "NIR": 842
        }
      closest_wavelengths = []


      for band, center in centers.items():

          idx = np.argmin(np.abs(wavelengths - center))

          closest_wavelengths.append(idx)

      indices = np.array(closest_wavelengths)

    elif hyperparameters['Bands'] == "G-RE705":
      centers = {
        # "R": 668,
        "G": 560,
        # "B": 475,
        "R_edge": 705,
        # "NIR": 842
        }
      closest_wavelengths = []


      for band, center in centers.items():

          idx = np.argmin(np.abs(wavelengths - center))

          closest_wavelengths.append(idx)

      indices = np.array(closest_wavelengths)

    elif hyperparameters['Bands'] == "G-RE740":
      centers = {
        # "R": 668,
        "G": 560,
        # "B": 475,
        "R_edge": 740,
        # "NIR": 842
        }
      closest_wavelengths = []


      for band, center in centers.items():

          idx = np.argmin(np.abs(wavelengths - center))

          closest_wavelengths.append(idx)

      indices = np.array(closest_wavelengths)

    elif hyperparameters['Bands'] == "R-RE740":
      centers = {
        "R": 668,
        # "G": 531,
        # "B": 475,
        "R_edge": 740,
        # "NIR": 842
        }
      closest_wavelengths = []


      for band, center in centers.items():

          idx = np.argmin(np.abs(wavelengths - center))

          closest_wavelengths.append(idx)

      indices = np.array(closest_wavelengths)


    elif hyperparameters['Bands'] == "R-RE705":
      centers = {
        "R": 668,
        # "G": 531,
        # "B": 475,
        "R_edge": 705,
        # "NIR": 842
        }
      closest_wavelengths = []


      for band, center in centers.items():

          idx = np.argmin(np.abs(wavelengths - center))

          closest_wavelengths.append(idx)

      indices = np.array(closest_wavelengths)

    elif hyperparameters['Bands'] == "R650-NIR":
      centers = {
        "R": 650,
        # "G": 531,
        # "B": 475,
        # "R_edge": 717,
        "NIR": 842
        }
      closest_wavelengths = []


      for band, center in centers.items():

          idx = np.argmin(np.abs(wavelengths - center))

          closest_wavelengths.append(idx)

      indices = np.array(closest_wavelengths)


    elif hyperparameters['Bands'] == "G531-NIR":
      centers = {
        # "R": 668,
        "G": 531,
        # "B": 475,
        # "R_edge": 717,
        "NIR": 842
        }
      closest_wavelengths = []


      for band, center in centers.items():

          idx = np.argmin(np.abs(wavelengths - center))

          closest_wavelengths.append(idx)

      indices = np.array(closest_wavelengths)

    elif hyperparameters['Bands'] == "B-NIR":
      centers = {
        # "R": 668,
        # "G": 560,
        "B": 475,
        # "R_edge": 717,
        "NIR": 842
        }
      closest_wavelengths = []


      for band, center in centers.items():

          idx = np.argmin(np.abs(wavelengths - center))

          closest_wavelengths.append(idx)

      indices = np.array(closest_wavelengths)


    elif hyperparameters['Bands'] == "G-NIR":
      centers = {
        # "R": 668,
        "G": 560,
        # "B": 475,
        # "R_edge": 717,
        "NIR": 842
        }
      closest_wavelengths = []


      for band, center in centers.items():

          idx = np.argmin(np.abs(wavelengths - center))

          closest_wavelengths.append(idx)

      indices = np.array(closest_wavelengths)


    elif hyperparameters['Bands'] == "R-NIR":
      centers = {
        "R": 668,
        # "G": 560,
        # "B": 475,
        # "R_edge": 717,
        "NIR": 842
        }
      closest_wavelengths = []


      for band, center in centers.items():

          idx = np.argmin(np.abs(wavelengths - center))

          closest_wavelengths.append(idx)

      indices = np.array(closest_wavelengths)

    elif hyperparameters['Bands'] == "R-RE717":
      centers = {
        "R": 668,
        # "G": 560,
        # "B": 475,
        "R_edge": 717,
        # "NIR": 842
        }
      closest_wavelengths = []


      for band, center in centers.items():

          idx = np.argmin(np.abs(wavelengths - center))

          closest_wavelengths.append(idx)

      indices = np.array(closest_wavelengths)

    elif hyperparameters['Bands'] == "G-RE717":
      centers = {
        # "R": 668,
        "G": 560,
        # "B": 475,
        "R_edge": 717,
        # "NIR": 842
        }
      closest_wavelengths = []


      for band, center in centers.items():

          idx = np.argmin(np.abs(wavelengths - center))

          closest_wavelengths.append(idx)

      indices = np.array(closest_wavelengths)


    elif hyperparameters['Bands'] == "B-RE717":
      centers = {
        # "R": 668,
        # "G": 560,
        "B": 475,
        "R_edge": 717,
        # "NIR": 842
        }
      closest_wavelengths = []


      for band, center in centers.items():

          idx = np.argmin(np.abs(wavelengths - center))

          closest_wavelengths.append(idx)

      indices = np.array(closest_wavelengths)

    elif hyperparameters['Bands'] == "3bands":
      centers = {
        "R": 668,
        "G": 560,
        "B": 475,
        # "R_edge": 717,
        # "NIR": 842
        }
      closest_wavelengths = []


      for band, center in centers.items():

          idx = np.argmin(np.abs(wavelengths - center))

          closest_wavelengths.append(idx)

      indices = np.array(closest_wavelengths)


    elif hyperparameters['Bands'] == "4bands":
      centers = {
        "R": 668,
        "G": 560,
        "B": 475,
        # "R_edge": 717,
        "NIR": 842
        }
      closest_wavelengths = []


      for band, center in centers.items():

          idx = np.argmin(np.abs(wavelengths - center))

          closest_wavelengths.append(idx)

      indices = np.array(closest_wavelengths)

    elif hyperparameters['Bands'] == "bandwidth":
      ranges = {
          "R": (660, 676),
          "G": (547, 573 ),
          "B": (459, 491),
          "R_edge": (711, 723),
          "NIR": (814, 870)
      }


      indices = np.concatenate([
          np.where((wavelengths >= rng[0]) & (wavelengths <= rng[1]))[0]
          for rng in ranges.values()
      ])
    elif hyperparameters['Bands'] == "MI":
      indices = np.array([134, 132, 128, 64, 125, 133, 123, 70, 62, 71, 131, 129, 74, 127, 67, 68, 66, 126, 69, 130])


    elif hyperparameters['Bands'] == "PCA":
      indices = np.array([  6, 190, 194, 3, 5, 196, 10, 171, 2, 18, 168, 195, 180, 11, 178, 192, 167, 191, 0, 179])


    print(indices.shape)
    print(indices)
    hyperparameters['indices'] = sorted(list(indices))

    if load_orig_images:
      base_dirs = [
          r"{0}/AronDataset/Field31.03.2022".format(hyperparameters['basedir']),
          r"{0}/AronDataset/Field06.04.2022".format(hyperparameters['basedir'])
      ]

      def find_closest_band(target_wavelength, wavelength_list):
        return np.abs(wavelength_list - target_wavelength).argmin()

      folder_path = r"{0}/Dark_Black_dark/raw_0".format(hyperparameters['basedir'])

      raw_path = folder_path
      hdr_path = folder_path+".hdr"
      dark_spectrum = envi.open(hdr_path, raw_path).load()
      dark_reference = np.repeat(dark_spectrum, 2, axis=0)


      for base_path in base_dirs:
          subfolders = [os.path.join(base_path, d) for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))]

          for folder in tqdm(subfolders):

              hdr_files = [f for f in os.listdir(folder) if f.endswith(".hdr")]
              key = f"{os.path.basename(base_path)}_{os.path.basename(folder)}"

              for hdr in hdr_files:
                  img_path = os.path.join(folder, hdr)

                  raw_file = hdr.replace(".hdr", "")
                  raw_path = os.path.join(folder, raw_file)
                  hdr_path = os.path.join(folder, hdr)

                  try:
                      img = envi.open(hdr_path, raw_path).load()
                  except Exception as e:
                      print(f"Skipping {hdr}: {e}")
                      continue

                  if os.path.basename(img_path) == "raw_0.hdr":
                      # red_idx = find_closest_band(650, wavelengths)
                      # green_idx = find_closest_band(550, wavelengths)
                      # blue_idx = find_closest_band(450, wavelengths)


                      # rgb_image = np.stack([
                      #     img[:, :, red_idx],
                      #     img[:, :, green_idx],
                      #     img[:, :, blue_idx]
                      # ], axis=-1)


                      # rgb_image = np.squeeze(rgb_image)


                      # rgb_image = rgb_image - np.min(rgb_image)
                      # rgb_image /= np.max(rgb_image)


                      # rgb_image_8bit = (rgb_image * 255).astype(np.uint8)
                      # cv2.imwrite("temp_rgb_image.png", cv2.cvtColor(rgb_image_8bit, cv2.COLOR_RGB2BGR))


                      # image_path = "temp_rgb_image.png"
                      # image = cv2.imread(image_path)


                      # gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)


                      # _, thresh = cv2.threshold(gray, 190, 255, cv2.THRESH_BINARY)


                      # contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)


                      # largest_contour = max(contours, key=cv2.contourArea)


                      # x, y, w, h = cv2.boundingRect(largest_contour)

                      # white_patch_hyperspectral = img[y+borders:y+h-borders, x+borders:x+w-borders, :]  # Shape: (h, w, fr)

                      # white_reference = np.mean(white_patch_hyperspectral, axis=(0, 1))  # Shape: (fr,)

                      # print(img_path)
                      # plt.figure(figsize=(8, 6))
                      # plt.imshow(rgb_image[y+borders:y+h-borders, x+borders:x+w-borders, :])
                      # plt.title("RGB white reference tile")
                      # plt.axis('off')
                      # plt.show()

                      #---- alternatively -----

                      target_wavelength = 728
                      band_728_idx = find_closest_band(target_wavelength, wavelengths)


                      gray_image_728nm = img[:, :, band_728_idx]


                      gray_image_728nm_normalized = cv2.normalize(gray_image_728nm, None, 0, 255, cv2.NORM_MINMAX)
                      gray_image_728nm_8bit = np.uint8(gray_image_728nm_normalized)


                      _, thresh = cv2.threshold(gray_image_728nm_8bit, 190, 255, cv2.THRESH_BINARY)


                      contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)


                      if contours:
                          largest_contour = max(contours, key=cv2.contourArea)


                          x, y, w, h = cv2.boundingRect(largest_contour)


                          white_patch_hyperspectral = img[y+borders:y+h-borders, x+borders:x+w-borders, :]


                          white_reference = np.mean(white_patch_hyperspectral, axis=(0, 1))


                          plt.figure(figsize=(8, 6))

                          visualization_img = cv2.cvtColor(gray_image_728nm_8bit, cv2.COLOR_GRAY2BGR)
                          cv2.rectangle(visualization_img, (x, y), (x+w, y+h), (0, 255, 0), 2)
                          plt.imshow(visualization_img)
                          plt.title("Detected White Reference on 728 nm Band")
                          plt.axis('off')
                          plt.show()

                      else:
                          print(f"Warning: No white reference tile found in {img_path}")

                          continue


                  img = (img - dark_reference) / (white_reference - dark_reference)

                  img = np.nan_to_num(img, nan=0.0, posinf=1.0, neginf=0.0)

                  global_max = img.max()
                  if global_max > 0:
                      img /= global_max

                  # Pixel per pixel normalization
                  min_val = np.min(img, axis=2, keepdims=True)
                  max_val = np.max(img, axis=2, keepdims=True)


                  img = (img - min_val) / (max_val - min_val)


                  for i in range(random_patches):
                    idx_height = np.random.randint(img.shape[0] - patch_size)
                    idx_width = np.random.randint(img.shape[1] - patch_size)



                    while (os.path.basename(img_path) == "raw_0.hdr") and idx_height >= y and idx_height <= y+h and idx_width >= x and idx_width <= x+w:
                      print("#----------------------#")
                      print("Reselecting patch")
                      print("#----------------------#")
                      idx_height = np.random.randint(img.shape[0] - patch_size)
                      idx_width = np.random.randint(img.shape[1] - patch_size)




                    training_patch = img[idx_height:idx_height+patch_size, idx_width:idx_width+patch_size, :]


                    window_length = 11  # must be odd
                    poly_order = 2
                    training_patch_smoothed = savgol_filter(training_patch, window_length, poly_order)

                    training_patch= np.transpose(training_patch_smoothed, (2, 0, 1))

                    saveFilename = f"{os.path.basename(base_path)}_{os.path.basename(folder)}_{hdr}"

                    training_str = "{0}/data/".format(hyperparameters['basedir'])+saveFilename.replace(".hdr", "")+"_training_"+str(i)


                    # Reflectance normalized
                    # plt.figure(figsize=(8, 4))
                    # plt.plot(wavelengths[hyperparameters['exclude_begin']:hyperparameters['total_length']-hyperparameters['exclude_end']],
                    #          training_patch[:,0, 0], label="Sample Spectrum")
                    # plt.xlabel("Wavelength (nm)")
                    # plt.ylabel("Reflectance")
                    # plt.title("Sample Spectrum")
                    # plt.legend()
                    # plt.grid()
                    # plt.show()


                    if os.path.exists(training_str+".npy"):
                        print("File exists.")
                    else:
                        np.save(training_str, np.array(training_patch))


              if load_all_images is False:
                  break


    if concatenate:
      directory = "{0}/data".format(hyperparameters['basedir'])

      training_data = []
      validation_data = []
      count = 0

      for filename in tqdm(os.listdir(directory)):
          if filename.endswith(".npy"):
              file_path = os.path.join(directory, filename)
              data = np.load(file_path)


              if "_training" in filename:
                  training_data.append(data)
              elif "_validation" in filename:
                  validation_data.append(data)

              if len(training_data) == 250: #and len(validation_data) == 240

                all_training_data = np.stack(training_data, axis=0)

                path = "{0}/T_data_{1}.npy".format(hyperparameters['basedir'],str(count))
                if os.path.exists(path):
                    print("File exists.")
                else:
                    np.save(path.replace(".npy", ""), all_training_data)
                count += 1
                training_data = []
                validation_data = []


    if concatenate_larger:

      input_arrays = [
          '{0}/T_data_0.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_1.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_2.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_3.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_4.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_5.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_6.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_7.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_8.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_9.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_10.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_11.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_12.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_13.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_14.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_15.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_16.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_17.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_18.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_19.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_20.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_21.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_22.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_23.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_24.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_25.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_26.npy'.format(hyperparameters['basedir']),
          '{0}/T_data_27.npy'.format(hyperparameters['basedir']),
      ]


      # Create concatenated memmaps
      hs_path, ms_path = create_concatenated_memmaps(
          input_arrays=input_arrays,
          output_dir='{0}/memmaps/{1}'.format(hyperparameters['basedir'], hyperparameters['Bands']),
          indices=hyperparameters['indices'],
          architecture=hyperparameters['architecture'],
          output_channels=hyperparameters['total_length']-(hyperparameters['exclude_begin']+hyperparameters['exclude_end']),
          hyperparameters=hyperparameters
      )
      print("All training data concatenated and saved.")


    hyperspectral_data_path = '{0}/memmaps/hyperspectral_full_{1}.dat'.format(hyperparameters['basedir'], hyperparameters['architecture'])
    multispectral_data_path = '{0}/memmaps/{1}/multispectral_selected_{2}.dat'.format(hyperparameters['basedir'],
                                                                                      hyperparameters['Bands'],
                                                                                      hyperparameters['architecture'])

    train_dataset = HyperspectralDataset(
      hyperspectral_path=hyperspectral_data_path,
      multispectral_path=multispectral_data_path,
      batch_size=hyperparameters['batch_size'],
      split='train',
      output_channels=hyperparameters['total_length']-(hyperparameters['exclude_begin']+hyperparameters['exclude_end']),
      data_percentage=hyperparameters['data_percentage'],
      indices=hyperparameters['indices'],
      architecture=hyperparameters['architecture']
    )

    val_dataset = HyperspectralDataset(
      hyperspectral_path=hyperspectral_data_path,
      multispectral_path=multispectral_data_path,
      batch_size=hyperparameters['batch_size'],
      split='val',
      output_channels=hyperparameters['total_length']-(hyperparameters['exclude_begin']+hyperparameters['exclude_end']),
      data_percentage=hyperparameters['data_percentage'],
      indices=hyperparameters['indices'],
      architecture=hyperparameters['architecture'],

    )

    test_dataset = HyperspectralDataset(
      hyperspectral_path=hyperspectral_data_path,
      multispectral_path=multispectral_data_path,
      batch_size=hyperparameters['batch_size'],
      split='test',
      output_channels=hyperparameters['total_length']-(hyperparameters['exclude_begin']+hyperparameters['exclude_end']),
      data_percentage=hyperparameters['data_percentage'],
      indices=hyperparameters['indices'],
      architecture=hyperparameters['architecture'],

    )
    train_dataloader = DataLoader(
        train_dataset,
        batch_size=None,
        num_workers=hyperparameters['num_workers']
    )

    val_dataloader = DataLoader(
        val_dataset,
        batch_size=None,
        num_workers=hyperparameters['num_workers']
    )

    test_dataloader = DataLoader(
        test_dataset,
        batch_size=None,
        num_workers=hyperparameters['num_workers']
    )


    now = str(datetime.now())
    now = now.replace(" ", "_")
    hyperparameters['datetime'] = now


    if hyperparameters_save:
      with open('{0}/hyperparameters/hyperparameters_{1}_{2}.pkl'.format(hyperparameters['basedir'], hyperparameters['name'], hyperparameters['datetime']), 'wb') as f:
          pickle.dump(hyperparameters, f)

    print('start training')
    train(dataloader=train_dataloader, val_dataloader=val_dataloader, test_dataloader=test_dataloader, hyperparameters=hyperparameters)

## 3. Load created generators from checkpoints
Functions: `hsi_cgan/evaluation/checkpoint_loading.py`.

In [ ]:
############################# ------------------ LOADING CREATED GENERATORS BLOCK  ------------------ #############################
bands_list = ['bandwidth', 'micasense', '3bands'] #'MI', 'PCA', 'bandwidth', 'micasense', '4bands', '3bands'
# 'B-RE717', 'R-RE717', 'G-RE717', 'G-NIR', 'B-NIR', 'R-NIR', 'G531-NIR', 'R650-NIR', 'R-RE705', 'R-RE740', 'B-RE705', 'B-RE740', 'G-RE705', 'G-RE740', 'G531-RE', 'R650-RE'
generators_list = ['ResNetGenerator', 'MLP'] #'CNNGenerator', 'ResNetGenerator','UNetGenerator', 'GeneratorLarge', 'MLP'


import os
import torch
import matplotlib.pyplot as plt
import numpy as np


dictionary_list = {'CNNGenerator':[0.001, 'rgb', 'PatchGANDiscriminator'],
                   'UNetGenerator':[0.001, 'rgb', 'PatchGANDiscriminator'],
                   'ResNetGenerator':[0.0001, 'rgb', 'PatchGANDiscriminator'],
                   'GeneratorLarge':[0.0001, 'MLP', 'DiscriminatorLarge'],
                   'MLP':[0.001, 'MLP', 'MLP'],
                   'MI': 20,
                   'PCA': 20,
                   'bandwidth': 56,
                   'micasense': 5,
                   '4bands': 4,
                   '3bands': 3,
                   'B-RE717': 2,
                   'R-RE717': 2,
                   'G-RE717': 2,
                   'G-NIR': 2,
                   'B-NIR': 2,
                   'R-NIR': 2,
                   'G531-NIR': 2,
                   'R650-NIR': 2,
                   'R-RE705': 2,
                   'R-RE740': 2,
                   'B-RE705': 2,
                   'B-RE740': 2,
                   'G-RE705': 2,
                   'G-RE740': 2,
                   'G531-RE': 2,
                   'R650-RE': 2,
                   }


hyperparameters = {
    'load_model': False,
    'hard_code_known': False,
    'data_percentage': 1,
    'num_workers': 2,
    # 'Bands': band, # MI, PCA, bandwidth, micasense
    'betas': (0.5, 0.999), #standard (0.5, 0.999)
    'batch_size': 16,
    'epochs': 100,
    # 'architecture': dictionary_list[generator][1], #options: rgb, cramer, MLP
    'name_loss': 'L1', #options: MSE, L1, BCE, CrossEntropy, SmoothL1Loss, KLDivLoss, HingeEmbeddingLoss, MarginRankingLoss, MultiLabelSoftMarginLoss, BCEWithLogitsLoss
    'name_adversarial_loss': 'BCEWithLogitsLoss',
    # 'name_g': generator, #options: CNNGenerator, UNetGenerator, ResNetGenerator, GeneratorLarge, MLP
    # 'name_d': dictionary_list[generator][2], #options: PatchGANDiscriminator, DiscriminatorLarge, MLP
    'name_optim': 'Adam', #options: Adam, SGD, RMSprop, Adagrad, Adadelta
    'checkpoint_epoch': 31,
    'basedir': '/content/drive/My Drive/AAA_thesis_twee',
    'decay_epochs': 25, #stepsize
    'decay_rate': 0.75, #gamma
    'exclude_begin': 35, #35
    'exclude_end': 40, #
}
wavelengths= np.load(r"{0}/wavelengthes.npy".format(hyperparameters['basedir']))
hyperparameters['total_length'] = len(wavelengths)

# Functions now live in hsi_cgan/evaluation/checkpoint_loading.py (this variant of
# creating_generator_and_discriminator / load_checkpoint overrides the training ones, as in the original notebook).
from hsi_cgan.evaluation.checkpoint_loading import (creating_generator_and_discriminator, load_checkpoint,
                                                 extract_model_name, plot_val_losses_from_checkpoints)
from hsi_cgan.evaluation import checkpoint_loading

# plot_val_losses_from_checkpoints reads these notebook globals; hand them to the module.
checkpoint_loading.configure(generators_list=generators_list, bands_list=bands_list,
                             dictionary_list=dictionary_list, hyperparameters=hyperparameters)


# Example usage:
checkpoint_directory = "/content/drive/My Drive/AAA_thesis_twee/models"
networks_created_list,MI_list, PCA_list, Bandwidth_list, Micasense_list, fourbands_list, threebands_list, MLP_list, Large_list, CNN_list, UNet_list, ResNet, B_RE717_list, R_RE717_list, G_RE717_list, G_NIR_list, B_NIR_list, R_NIR_list, G531_NIR_list, R650_NIR_list, R_RE705_list, R_RE740_list, B_RE705_list, B_RE740_list, G_RE705_list, G_RE740_list, G531_RE_list, R650_RE_list = plot_val_losses_from_checkpoints(checkpoint_directory)

## 4. Evaluation functions
Imports from `hsi_cgan/metrics`, `hsi_cgan/evaluation/validators.py`, `hsi_cgan/visualization/spectral_plots.py`.

In [ ]:
############################# ------------------ FUNCTION MAKING FOR RMSE ETC BLOCK  ------------------#############################

from hsi_cgan.metrics.losses import rmse_loss, mrae_loss
from hsi_cgan.metrics.image_metrics import sam_loss, calculate_metrics, calculate_per_channel_metrics
from hsi_cgan.evaluation.validators import validate, validate_extended, validate_per_channel
from hsi_cgan.visualization.spectral_plots import plot_spectral_curves, visualize_validation_results
from hsi_cgan.visualization import spectral_plots

# plot_spectral_curves reads the notebook global `hyperparameters`; hand it to the module.
spectral_plots.configure(hyperparameters=hyperparameters)

## 5. Calculate RMSE etc. on the test set

In [ ]:
############################# ------------------ STARTING CALCULATION OF RMSE ETC BLOCK  ------------------#############################
import json
bands_list = ['3bands', 'bandwidth', 'micasense', 'B-RE717', 'R-RE717', 'G-RE717', 'G-NIR', 'B-NIR', 'R-NIR', 'G531-NIR', 'R650-NIR', 'R-RE705', 'R-RE740', 'B-RE705', 'B-RE740', 'G-RE705', 'G-RE740', 'G531-RE', 'R650-RE'] #'MI', 'PCA', 'bandwidth', 'micasense'

generators_list = ['ResNetGenerator', 'MLP'] #'CNNGenerator', 'UNetGenerator', , 'GeneratorLarge', 'MLP', ResNetGenerator





indices_3bands = np.array([ 86, 38, 0])

indices_4bands = np.array([ 86, 38, 0, 164])

indices_micasense = np.array([ 0, 38, 86, 108, 164])


indices_bandwidth = np.array([ 83,  84,  85,  86,  87,  88,  89,  32,  33,  34,  35,  36,  37, 38,  39,  40,  41,  42,
  43,   0,   1,   2,   3,   4,   5,   6, 106, 107, 108, 109, 110, 152, 153, 154, 155, 156,
 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174,
 175, 176])


indices_MI = np.array([134, 132, 128, 64, 125, 133, 123, 70, 62, 71, 131, 129, 74, 127, 67, 68, 66, 126, 69, 130])


indices_PCA = np.array([  6, 190, 194, 3, 5, 196, 10, 171, 2, 18, 168, 195, 180, 11, 178, 192, 167, 191, 0, 179])

#----------------
indices_B_RE717 = np.array([ 0, 108])

indices_R_RE717 = np.array([86, 108])

indices_G_RE717 = np.array([ 38, 108])

indices_G_NIR = np.array([ 38, 164])

indices_B_NIR = np.array([ 0, 164])

indices_R_NIR = np.array([86, 164])

indices_G531_NIR = np.array([ 25, 164])

indices_R650_NIR = np.array([78, 164])

indices_R_RE705 = np.array([86, 103])

indices_R_RE740 = np.array([86, 118])

indices_B_RE705 = np.array([ 0, 103])

indices_B_RE740 = np.array([ 0, 118])

indices_G_RE705 = np.array([ 38, 103])

indices_G_RE740 = np.array([ 38, 118])

indices_G531_RE = np.array([ 25, 108])

indices_R650_RE = np.array([78, 108])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

wavelengths= np.load(r"{0}/wavelengthes.npy".format(hyperparameters['basedir']))
wavelengths =wavelengths[35:272-40]

dict_names= {'CNN': ['rgb', 'rgb'],
             'ResNet':['rgb', 'rgb'],
             'U-Net': ['rgb', 'rgb'],
             'Cramér':['rgb', 'MLP'],
             'MLP':['rgb', 'MLP'],
             'MI':['MI', indices_MI],
             'PCA':['PCA', indices_PCA],
             'Centers + Bandwidth':['bandwidth', indices_bandwidth],
             'Centers':['micasense', indices_micasense],
             'RGB + NIR':['4bands', indices_4bands],
             'RGB':['3bands', indices_3bands],
              'B-RE717': ['B-RE717', indices_B_RE717],
              'R-RE717': ['R-RE717', indices_R_RE717],
              'G-RE717': ['G-RE717', indices_G_RE717],
              'G-NIR': ['G-NIR', indices_G_NIR],
              'B-NIR': ['B-NIR', indices_B_NIR],
              'R-NIR': ['R-NIR', indices_R_NIR],
              'G531-NIR': ['G531-NIR', indices_G531_NIR],
              'R650-NIR': ['R650-NIR', indices_R650_NIR],
              'R-RE705': ['R-RE705', indices_R_RE705],
              'R-RE740': ['R-RE740', indices_R_RE740],
              'B-RE705': ['B-RE705', indices_B_RE705],
              'B-RE740': ['B-RE740', indices_B_RE740],
              'G-RE705': ['G-RE705', indices_G_RE705],
              'G-RE740': ['G-RE740', indices_G_RE740],
              'G531-RE': ['G531-RE', indices_G531_RE],
              'R650-RE': ['R650-RE', indices_R650_RE],


             }




for i in range(len(networks_created_list)):
  name, generator, discriminator = networks_created_list[i]
  print(name)

  output = name.split(" ", 1)

  print(output)


  hyperparameters['architecture'] = dict_names[output[0]][0]
  print("Indices:", dict_names[output[1]][1])
  hyperparameters['Bands'] = dict_names[output[1]][0]
  hyperparameters['indices'] = dict_names[output[1]][1]



  hyperspectral_data_path = '{0}/memmaps/hyperspectral_full_{1}.dat'.format(hyperparameters['basedir'], hyperparameters['architecture'])
  multispectral_data_path = '{0}/memmaps/{1}/multispectral_selected_{2}.dat'.format(hyperparameters['basedir'],
                                                                                    hyperparameters['Bands'],
                                                                                    hyperparameters['architecture'])

  train_dataset = HyperspectralDataset(
    hyperspectral_path=hyperspectral_data_path,
    multispectral_path=multispectral_data_path,
    batch_size=hyperparameters['batch_size'],
    split='train',
    output_channels=hyperparameters['total_length']-(hyperparameters['exclude_begin']+hyperparameters['exclude_end']),
    data_percentage=hyperparameters['data_percentage'],
    indices=hyperparameters['indices'],
    architecture=hyperparameters['architecture']
  )

  val_dataset = HyperspectralDataset(
    hyperspectral_path=hyperspectral_data_path,
    multispectral_path=multispectral_data_path,
    batch_size=hyperparameters['batch_size'],
    split='val',
    output_channels=hyperparameters['total_length']-(hyperparameters['exclude_begin']+hyperparameters['exclude_end']),
    data_percentage=hyperparameters['data_percentage'],
    indices=hyperparameters['indices'],
    architecture=hyperparameters['architecture'],

  )

  test_dataset = HyperspectralDataset(
    hyperspectral_path=hyperspectral_data_path,
    multispectral_path=multispectral_data_path,
    batch_size=hyperparameters['batch_size'],
    split='test',
    output_channels=hyperparameters['total_length']-(hyperparameters['exclude_begin']+hyperparameters['exclude_end']),
    data_percentage=hyperparameters['data_percentage'],
    indices=hyperparameters['indices'],
    architecture=hyperparameters['architecture'],

  )
  train_dataloader = DataLoader(
      train_dataset,
      batch_size=None,
      num_workers=hyperparameters['num_workers']
  )

  val_dataloader = DataLoader(
      val_dataset,
      batch_size=None,
      num_workers=hyperparameters['num_workers']
  )

  test_dataloader = DataLoader(
      test_dataset,
      batch_size=None,
      num_workers=hyperparameters['num_workers']
  )




  # print(f"\n Model name: {name}")
  generator.to(device)
  results = validate(generator, test_dataloader, device, dict_names[output[0]][1], len(hyperparameters['indices']))

  print(f"Average RMSE: {results['RMSE']:.4f}")
  print(f"Average MRAE: {results['MRAE']:.4f}")
  print(f"Average SAM: {results['SAM']:.4f}")
  print(f"Average PSNR: {results['PSNR']:.4f}")
  print(f"Average SSIM: {results['SSIM']:.4f}")

  # results = validate_per_channel(generator, test_dataloader, device, dict_names[output[0]][1], len(hyperparameters['indices']))
  # results_file_path = f"{hyperparameters['basedir']}/results_{name}.json"  # Create a filename based on the model name
  # with open(results_file_path, 'w') as results_file:
  #   json.dump(results, results_file, indent=4)  # Save results as JSON with indentation

  # # print(f"Results saved to: {results_file_path}")
  # # visualize_metric_div_variance_results(results, wavelengths, len(test_dataloader), hyperparameters['indices'])
  # visualize_validation_results(results, wavelengths, len(test_dataloader), hyperparameters['indices'])


  # results = validate_extended(generator, test_dataloader, device, dict_names[output[0]][1], hyperparameters['indices'])
  # print(f"Spectral RMSE: {results['spectral_rmse']} ± {np.mean(results['true_ci'])}")
  # print(f"Spectral MRAE: {results['spectral_mrae']} ± {np.mean(results['true_ci'])}")
  # print(f"Spectral RMSE (pred ci): {results['spectral_rmse']} ± {np.mean(results['pred_ci'])}")
  # print(f"Spectral MRAE (pred ci): {results['spectral_mrae']} ± {np.mean(results['pred_ci'])}")
  # print(f"Pixel RMSE: {results['pixel_rmse']} ")
  # print(f"Pixel MRAE: {results['pixel_mrae']} ")




## 6. Vegetation index images
Functions: `hsi_cgan/evaluation/vegetation_indices.py`, `hsi_cgan/visualization/vegetation_plots.py`.

In [ ]:
############################# ------------------ VEGETATION INDICES IMAGES BLOCK  ------------------#############################

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from skimage.metrics import structural_similarity
import matplotlib.gridspec as gridspec


from hsi_cgan.evaluation.vegetation_indices import calculate_vegetation_index
from hsi_cgan.visualization.vegetation_plots import plot_vegetation_index_comparison


indices_3bands = np.array([ 86, 38, 0])

indices_4bands = np.array([ 86, 38, 0, 164])

indices_micasense = np.array([ 86, 38, 0, 108, 164])


indices_bandwidth = np.array([ 83,  84,  85,  86,  87,  88,  89,  32,  33,  34,  35,  36,  37, 38,  39,  40,  41,  42,
  43,   0,   1,   2,   3,   4,   5,   6, 106, 107, 108, 109, 110, 152, 153, 154, 155, 156,
 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174,
 175, 176])


indices_MI = np.array([134, 132, 128, 64, 125, 133, 123, 70, 62, 71, 131, 129, 74, 127, 67, 68, 66, 126, 69, 130])


indices_PCA = np.array([  6, 190, 194, 3, 5, 196, 10, 171, 2, 18, 168, 195, 180, 11, 178, 192, 167, 191, 0, 179])

#----------------
indices_B_RE717 = np.array([ 0, 108])

indices_R_RE717 = np.array([86, 108])

indices_G_RE717 = np.array([ 38, 108])

indices_G_NIR = np.array([ 38, 164])

indices_B_NIR = np.array([ 0, 164])

indices_R_NIR = np.array([86, 164])

indices_G531_NIR = np.array([ 25, 164])

indices_R650_NIR = np.array([78, 164])

indices_R_RE705 = np.array([86, 103])

indices_R_RE740 = np.array([86, 118])

indices_B_RE705 = np.array([ 0, 103])

indices_B_RE740 = np.array([ 0, 118])

indices_G_RE705 = np.array([ 38, 103])

indices_G_RE740 = np.array([ 38, 118])

indices_G531_RE = np.array([ 25, 108])

indices_R650_RE = np.array([78, 108])


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

wavelengths= np.load(r"{0}/wavelengthes.npy".format(hyperparameters['basedir']))
wavelengths =wavelengths[35:272-40]

dict_names= {
            'CNNGenerator': ['rgb', 'rgb'],
             'ResNet':['rgb', 'rgb'],
             'UNetGenerator': ['rgb', 'rgb'],
             'GeneratorLarge':['rgb', 'MLP'],
             'MLP':['rgb', 'MLP'],
             'MI':['MI', indices_MI],
             'PCA':['PCA', indices_PCA],
             'Centers + Bandwidth':['bandwidth', indices_bandwidth],
             'Centers':['micasense', indices_micasense],
             '4bands':['4bands', indices_4bands],
             'RGB':['3bands', indices_3bands],
              'B-RE717': ['B-RE171', indices_B_RE717],
              'R-RE717': ['R-RE717', indices_R_RE717],
              'G-RE717': ['G-RE717', indices_G_RE717],
              'G-NIR': ['G-NIR', indices_G_NIR],
              'B-NIR': ['B-NIR', indices_B_NIR],
              'R-NIR': ['R-NIR', indices_R_NIR],
              'G531-NIR': ['G531-NIR', indices_G531_NIR],
              'R650-NIR': ['R650-NIR', indices_R650_NIR],
              'R-RE705': ['R-RE705', indices_R_RE705],
              'R-RE740': ['R-RE740', indices_R_RE740],
              'B-RE705': ['B-RE705', indices_B_RE705],
              'B-RE740': ['B-RE740', indices_B_RE740],
              'G-RE705': ['G-RE705', indices_G_RE705],
              'G-RE740': ['G-RE740', indices_G_RE740],
              'G531-RE': ['G531-RE', indices_G531_RE],
              'R650-RE': ['R650-RE', indices_R650_RE],
             }


for i in range(len(networks_created_list)):
  name, generator, discriminator = networks_created_list[i]
  print(name)
  model, scenario = name.split(' ', 1)

  hyperparameters['architecture'] = dict_names[model][1]
  hyperparameters['Bands'] = dict_names[scenario][0]
  hyperparameters['indices'] = dict_names[scenario][1]

  hyperspectral_data_path = '{0}/memmaps/hyperspectral_full_{1}.dat'.format(hyperparameters['basedir'], 'rgb')
  multispectral_data_path = '{0}/memmaps/{1}/multispectral_selected_{2}.dat'.format(hyperparameters['basedir'],
                                                                                    hyperparameters['Bands'],
                                                                                    'rgb')

  train_dataset = HyperspectralDataset(
    hyperspectral_path=hyperspectral_data_path,
    multispectral_path=multispectral_data_path,
    batch_size=hyperparameters['batch_size'],
    split='train',
    output_channels=hyperparameters['total_length']-(hyperparameters['exclude_begin']+hyperparameters['exclude_end']),
    data_percentage=hyperparameters['data_percentage'],
    indices=hyperparameters['indices'],
    architecture='rgb'
  )

  val_dataset = HyperspectralDataset(
    hyperspectral_path=hyperspectral_data_path,
    multispectral_path=multispectral_data_path,
    batch_size=hyperparameters['batch_size'],
    split='val',
    output_channels=hyperparameters['total_length']-(hyperparameters['exclude_begin']+hyperparameters['exclude_end']),
    data_percentage=hyperparameters['data_percentage'],
    indices=hyperparameters['indices'],
    architecture='rgb',

  )

  test_dataset = HyperspectralDataset(
    hyperspectral_path=hyperspectral_data_path,
    multispectral_path=multispectral_data_path,
    batch_size=hyperparameters['batch_size'],
    split='test',
    output_channels=hyperparameters['total_length']-(hyperparameters['exclude_begin']+hyperparameters['exclude_end']),
    data_percentage=hyperparameters['data_percentage'],
    indices=hyperparameters['indices'],
    architecture='rgb',

  )
  train_dataloader = DataLoader(
      train_dataset,
      batch_size=None,
      num_workers=hyperparameters['num_workers']
  )

  val_dataloader = DataLoader(
      val_dataset,
      batch_size=None,
      num_workers=hyperparameters['num_workers']
  )

  test_dataloader = DataLoader(
      test_dataset,
      batch_size=None,
      num_workers=hyperparameters['num_workers']
  )


  sample_msi, real_hsi = next(iter(test_dataloader))
  real_hsi = real_hsi.numpy()[0, :, :, :]
  real_hsi = np.transpose(real_hsi, (1, 2, 0))

  generator.to(device)

  if model == 'MLP':
    B, C, H, W = sample_msi.shape
    print(sample_msi.shape)

    sample_msi = sample_msi.permute(2, 3, 0, 1).reshape(-1, C)

    fake_hs_images = generator(sample_msi.to(device))
    print(fake_hs_images.shape)

    C_out = fake_hs_images.shape[-1]
    predicted_hsi = fake_hs_images.reshape(H, W, B, C_out).permute(2, 3, 0, 1).cpu().detach().numpy()[0, :, :, :]
    print(predicted_hsi.shape)
  else:
    predicted_hsi = generator(sample_msi.to(device)).cpu().detach().numpy()[0, :, :, :]

  predicted_hsi = np.transpose(predicted_hsi, (1, 2, 0))

  wavelengths= np.load(r"{0}/wavelengthes.npy".format(hyperparameters['basedir']))
  wavelengths =wavelengths[35:272-40]

  plot_vegetation_index_comparison(real_hsi, predicted_hsi, wavelengths)

## 7. Loss plots: 16 two-band scenarios
Functions: `hsi_cgan/visualization/band_plots.py`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


from hsi_cgan.visualization.band_plots import plot_specific_band_generator_and_discriminator


plot_specific_band_generator_and_discriminator(
    B_RE717_list, R_RE717_list, G_RE717_list,
    G_NIR_list, B_NIR_list, R_NIR_list,
    G531_NIR_list, R650_NIR_list, R_RE705_list,
    R_RE740_list, B_RE705_list, B_RE740_list,
    G_RE705_list, G_RE740_list, G531_RE_list, R650_RE_list
)

print("Plot saved as specific_band_generator_discriminator_loss.png")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from hsi_cgan.visualization.band_plots import plot_specific_band_comparisons


plot_specific_band_comparisons(
    B_RE717_list, R_RE717_list, G_RE717_list,
    G_NIR_list, B_NIR_list, R_NIR_list,
    G531_NIR_list, R650_NIR_list, R_RE705_list,
    R_RE740_list, B_RE705_list, B_RE740_list,
    G_RE705_list, G_RE740_list, G531_RE_list, R650_RE_list
)

print("Plot saved as specific_band_comparisons.png")

## 8. Loss plots: band types and generator architectures
Functions: `hsi_cgan/visualization/band_plots.py`, `hsi_cgan/visualization/architecture_plots.py`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from hsi_cgan.visualization.band_plots import plot_band_comparisons
from hsi_cgan.visualization.architecture_plots import plot_generator_comparisons, plot_generator_and_discriminator


plot_band_comparisons(MI_list, PCA_list, Bandwidth_list, Micasense_list, fourbands_list, threebands_list)
plot_generator_comparisons(MLP_list, Large_list, CNN_list, UNet_list, ResNet)
plot_generator_and_discriminator(MLP_list, Large_list, CNN_list, UNet_list, ResNet)